[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/15_Fitting_Likelihood.ipynb)

# Fitting II: Maximum Likelihood and Toy Monte Carlo

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 12

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

Last week you fitted data by minimizing $\chi^2$, and you ended on a problem: when a histogram has bins with few or zero counts, $\chi^2$ with $\sqrt{n}$ errors breaks down. This week you learn the more general and more powerful method that particle and nuclear physicists use for almost every measurement: **maximum likelihood**. You will also learn how to *test* a fitting method by running thousands of simulated experiments ("toy Monte Carlo"), and how to get uncertainties by resampling your own data (the bootstrap).

The running example is the **muon lifetime**. *Physics recap:* the muon is an unstable cousin of the electron. At rest it decays ($\mu \to e\,\nu\,\bar\nu$) with mean lifetime $\tau_\mu = 2.197\ \mu$s, so its decay time $t$ follows the exponential PDF $p(t) = e^{-t/\tau}/\tau$ that you sampled in the Monte Carlo simulation notebook. All data in this notebook are **simulated**, so you always know the right answer. This connects to Project 3.

## Learning goals

By the end of this notebook you should be able to:

- define the likelihood and explain why we minimize $-2\ln L$;
- derive the maximum-likelihood estimate of a lifetime and its uncertainty;
- minimize $-\ln L$ numerically with `scipy.optimize.minimize`, and get uncertainties from a likelihood scan ($\Delta(-2\ln L) = 1$) and from the curvature (Hessian);
- compare unbinned likelihood, binned $\chi^2$, and binned Poisson likelihood fits, and explain why $\chi^2$ is biased at low counts;
- fit a signal plus background model without binning;
- run toy Monte Carlo experiments to check bias, pulls, and coverage;
- estimate uncertainties with the bootstrap;
- judge goodness of fit and compare models with $\Delta\ln L$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, curve_fit, brentq
from scipy.stats import chi2, norm

rng = np.random.default_rng(3)
TAU_MU = 2.197      # true muon lifetime in microseconds (used to simulate data)

## 1. The likelihood

Suppose you measured decay times $t_1, t_2, \dots, t_N$ and want to know $\tau$. For a trial value of $\tau$, the probability density of getting one particular time $t_i$ is $p(t_i\,|\,\tau)$ (read the bar as "given"). Different decays are **independent**, and for independent events probabilities multiply. So the probability density of observing the whole data set is

$$L(\tau) = \prod_{i=1}^{N} p(t_i\,|\,\tau) .$$

Viewed as a function of the *parameter* $\tau$ with the data held fixed, this is called the **likelihood**. The **maximum-likelihood (ML) principle**: the best estimate of $\tau$ is the value that makes the observed data most probable, i.e. that maximizes $L$.

A tiny example with three decay times makes this concrete.

In [ ]:
def p_exp(t, tau):
    """Exponential decay-time PDF."""
    return np.exp(-t / tau) / tau

times3 = np.array([0.8, 1.9, 3.5])      # microseconds
for tau_try in [1.0, 2.0, 3.0, 5.0]:
    L = np.prod(p_exp(times3, tau_try))  # np.prod multiplies all elements
    print(f"tau = {tau_try:.1f} us:  L = {L:.5f}")

$L$ is largest for $\tau$ around 2, close to the average of the three times. Too short a $\tau$ makes the long time 3.5 $\mu$s improbable; too long a $\tau$ spreads the probability too thin.

## 2. Why use $-2 \ln L$?

Three practical reasons:

1. **Numbers.** A product of many small densities underflows: with a thousand events, $L$ is smaller than the smallest positive floating-point number and becomes exactly 0.
2. **Sums are easier than products.** $\ln L = \sum_i \ln p(t_i\,|\,\tau)$. Since $\ln$ is increasing, maximizing $\ln L$ is the same as maximizing $L$.
3. **Minimizers minimize.** Optimization routines find minima, so we minimize the **negative log-likelihood** $-\ln L$ (NLL). The extra factor of 2 makes it directly comparable to $\chi^2$ (see below).

In [ ]:
t_big = rng.exponential(TAU_MU, 1000)
print("L     =", np.prod(p_exp(t_big, TAU_MU)))          # underflows to 0.0
print("ln L  =", np.sum(np.log(p_exp(t_big, TAU_MU))))   # perfectly fine

### Connection to least squares

If each measurement $y_i$ has a Gaussian error $\sigma_i$ around the model $f(x_i)$, then

$$-2\ln L = -2\sum_i \ln\left[\frac{1}{\sqrt{2\pi}\sigma_i} e^{-(y_i - f(x_i))^2/2\sigma_i^2}\right] = \sum_i \left(\frac{y_i - f(x_i)}{\sigma_i}\right)^2 + \text{const} = \chi^2 + \text{const}.$$

So **least squares is maximum likelihood for Gaussian errors**. Maximum likelihood is the more general principle: it works for any PDF, including exponential decay times and Poisson counts, where the Gaussian assumption fails.

## 3. The lifetime, analytically

For the exponential PDF,

$$\ln L(\tau) = \sum_{i=1}^N \left(-\ln\tau - \frac{t_i}{\tau}\right) = -N\ln\tau - \frac{1}{\tau}\sum_i t_i .$$

Set the derivative to zero:

$$\frac{d\ln L}{d\tau} = -\frac{N}{\tau} + \frac{1}{\tau^2}\sum_i t_i = 0 \quad\Longrightarrow\quad \hat\tau = \frac{1}{N}\sum_i t_i = \bar t .$$

The ML estimate of the lifetime is simply the **mean decay time**. (The hat means "estimate of.")

**Uncertainty from the curvature.** Near its maximum, $\ln L$ is approximately a downward parabola. Writing it as $\ln L(\tau) \approx \ln L_\text{max} - \frac{(\tau - \hat\tau)^2}{2\sigma^2}$ (compare the exponent of a Gaussian), the width $\sigma$ is related to the second derivative by

$$\frac{1}{\sigma^2} = -\frac{d^2 \ln L}{d\tau^2}\bigg|_{\hat\tau}.$$

A sharply curved likelihood means a precise measurement. For the exponential, $\frac{d^2\ln L}{d\tau^2} = \frac{N}{\tau^2} - \frac{2}{\tau^3}\sum_i t_i$, which at $\hat\tau = \bar t$ equals $-N/\hat\tau^2$, so

$$\sigma_{\hat\tau} = \frac{\hat\tau}{\sqrt N}.$$

The relative precision improves like $1/\sqrt N$, as you have come to expect from Monte Carlo.

Let's simulate a small experiment: 50 muon decays.

In [ ]:
N = 50
t_data = rng.exponential(TAU_MU, N)          # simulated decay times (us)

tau_hat = t_data.mean()
sigma_tau = tau_hat / np.sqrt(N)
print(f"tau_hat = {tau_hat:.3f} +/- {sigma_tau:.3f} us   (true {TAU_MU} us)")

### Uncertainty from a likelihood scan

The parabola picture also gives a graphical rule. Define $\Delta(-2\ln L) = -2\ln L(\tau) - (-2\ln L)_\text{min}$. With the parabola above, $\Delta(-2\ln L) = (\tau - \hat\tau)^2/\sigma^2$, which equals **1** at $\tau = \hat\tau \pm \sigma$. So:

> Scan $-2\ln L$; the $1\sigma$ interval is where it lies within 1 of its minimum.

This is the same rule as $\Delta\chi^2 = 1$ last week. When the likelihood is not a perfect parabola (small $N$), the scan gives an **asymmetric** interval, which is more accurate than the parabola.

We find the crossing points with `brentq` from the root-finding notebook.

In [ ]:
def nll_exp(tau, t):
    """Negative log-likelihood -ln L for exponential decay times t."""
    return np.sum(t / tau + np.log(tau))

tau_grid = np.linspace(1.4, 3.6, 400)
m2lnL = 2 * np.array([nll_exp(tg, t_data) for tg in tau_grid])
d_m2lnL = m2lnL - 2 * nll_exp(tau_hat, t_data)

f_cross = lambda tg: 2 * (nll_exp(tg, t_data) - nll_exp(tau_hat, t_data)) - 1
lo_1s = brentq(f_cross, 0.5 * tau_hat, tau_hat)
hi_1s = brentq(f_cross, tau_hat, 3 * tau_hat)
print(f"scan interval: {lo_1s:.3f} to {hi_1s:.3f} us  ->  tau = {tau_hat:.3f} -{tau_hat - lo_1s:.3f} +{hi_1s - tau_hat:.3f}")
print(f"parabola:      +/- {sigma_tau:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tau_grid, d_m2lnL, label=r"$\Delta(-2\ln L)$")
ax.plot(tau_grid, (tau_grid - tau_hat)**2 / sigma_tau**2, "--", label="parabola (curvature)")
ax.axhline(1, color="k", lw=0.8)
ax.axvspan(lo_1s, hi_1s, color="C2", alpha=0.2, label=r"$1\sigma$ interval from scan")
ax.set_ylim(0, 5)
ax.set_xlabel(r"$\tau$ ($\mu$s)")
ax.set_ylabel(r"$\Delta(-2\ln L)$")
ax.legend()
plt.show()

The scan is slightly lopsided: long lifetimes are a bit harder to rule out than short ones. With 50 events the difference is small; with 5 events it is large (Your turn 2).

### Your turn 1

*Counting experiment.* A Geiger counter records the following numbers of counts in ten 1-minute intervals: `[3, 6, 2, 4, 5, 3, 7, 4, 2, 5]`. Each count follows a Poisson distribution with unknown mean $\lambda$: $P(k\,|\,\lambda) = \lambda^k e^{-\lambda}/k!$.

1. Write $-\ln L(\lambda)$ for the ten counts. (You may drop the $\ln k!$ terms: they do not depend on $\lambda$, so they do not move the minimum.)
2. Show on paper that $\hat\lambda$ is the mean count, then confirm by a scan.
3. Find the $1\sigma$ interval with the $\Delta(-2\ln L) = 1$ rule and compare with $\sqrt{\hat\lambda/10}$.

In [ ]:
# Your code here

## 4. Numerical minimization with `scipy.optimize.minimize`

For most models you cannot solve $d\ln L/d\theta = 0$ by hand. `scipy.optimize.minimize(fun, x0)` finds a minimum of a function `fun` of a parameter **array**, starting from the guess `x0`. It returns a result object with fields:

- `res.x`: the parameters at the minimum (an array),
- `res.fun`: the function value there,
- `res.success`: whether it believes it converged, and `res.message`.

We use the **Nelder–Mead** method, which needs only function values (no derivatives) and is robust for a few parameters. Its `bounds` option keeps parameters in a sensible range, here $\tau > 0$ so that $\ln\tau$ is defined.

In [ ]:
res = minimize(lambda p: nll_exp(p[0], t_data), x0=[1.0], method="Nelder-Mead",
               bounds=[(0.01, 50)])
print(res.message)
print(f"numerical tau_hat = {res.x[0]:.4f} us;  analytic (mean) = {tau_hat:.4f} us")

### Uncertainty from the Hessian

The **Hessian** is the matrix of second derivatives of a function of several parameters, $H_{ij} = \partial^2 f/\partial\theta_i\partial\theta_j$. For one parameter it is just the second derivative. You can compute it with the central-difference formulas from the numerical differentiation notebook. For a mixed derivative the formula is

$$\frac{\partial^2 f}{\partial\theta_i \partial\theta_j} \approx \frac{f(+h_i, +h_j) - f(+h_i, -h_j) - f(-h_i, +h_j) + f(-h_i, -h_j)}{4 h_i h_j},$$

where $f(+h_i, -h_j)$ means $f$ evaluated with $\theta_i$ shifted by $+h_i$ and $\theta_j$ by $-h_j$. (For $i = j$ it reduces to the familiar second-difference formula with step $2h$.)

For one parameter, $\sigma^2 = 1/H$ when $f = -\ln L$. For several parameters, the generalization is that the **covariance matrix is the inverse of the Hessian of $-\ln L$**: $C = H^{-1}$.

*What is a matrix inverse?* For a square matrix $H$, the inverse $H^{-1}$ is the matrix that "undoes" it, the way $1/x$ undoes multiplication by $x$: $H H^{-1} = I$, where the identity matrix $I$ has 1s on the diagonal and 0s elsewhere. NumPy computes it with `np.linalg.inv(H)`. For a $1\times1$ matrix it is just $1/H$, giving back $\sigma^2 = 1/H$.

In [ ]:
def hessian(f, x, h):
    """Numerical Hessian of f at x by central differences; h = array of step sizes."""
    x = np.asarray(x, dtype=float)
    n = x.size
    H = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n); ei[i] = h[i]
            ej = np.zeros(n); ej[j] = h[j]
            H[i, j] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * h[i] * h[j])
    return H

H = hessian(lambda p: nll_exp(p[0], t_data), res.x, h=[1e-3])
cov = np.linalg.inv(H)
print("Hessian:", H, "  covariance:", cov)
print(f"sigma from Hessian = {np.sqrt(cov[0, 0]):.4f} us;  analytic tau_hat/sqrt(N) = {sigma_tau:.4f} us")

Three ways, one answer (up to the small asymmetry the scan captures).

**Pitfall.** With the default method (BFGS), `minimize` also returns `res.hess_inv`, an *approximate* inverse Hessian built up during the search. It can be quite inaccurate; do not use it for final uncertainties without checking.

### Your turn 2

Draw a new data set of only $N = 5$ decays (use `rng.exponential`). Find $\hat\tau$, the scan interval, and the Hessian uncertainty. How asymmetric is the scan interval now? Plot $\Delta(-2\ln L)$ and the parabola on the same axes.

In [ ]:
# Your code here

## 5. Unbinned vs binned fits at low statistics

What we did above is an **unbinned** fit: each event enters the likelihood individually, so no information is lost to binning. Often, though, data come as a histogram. Let's compare three ways to fit a histogram of 100 simulated muon decays, using 40 bins of 0.3 $\mu$s from 0 to 12 $\mu$s:

1. **Unbinned ML** on the individual times (the mean, as above).
2. **Binned $\chi^2$** with $\sigma_i = \sqrt{n_i}$, dropping empty bins (the "usual" approach from last week).
3. **Binned Poisson likelihood.** Each bin count $n_i$ is a Poisson random number with mean $\mu_i(\theta)$ predicted by the model, so

$$-\ln L = -\sum_i \ln\frac{\mu_i^{n_i} e^{-\mu_i}}{n_i!} = \sum_i \big(\mu_i - n_i\ln\mu_i\big) + \text{const}.$$

Empty bins are no problem: they contribute just $\mu_i$.

For the binned fits the model is the expected count in each bin. With $A$ the total number of decays (a free normalization) and bin edges $a_i, b_i$, integrating the exponential PDF over the bin gives

$$\mu_i = A\left(e^{-a_i/\tau} - e^{-b_i/\tau}\right).$$

In [ ]:
edges = np.linspace(0, 12, 41)
centers = 0.5 * (edges[:-1] + edges[1:])

def mu_bins(A, tau):
    """Expected counts per bin for A decays with lifetime tau."""
    return A * (np.exp(-edges[:-1] / tau) - np.exp(-edges[1:] / tau))

def fit_three_ways(t):
    """Return the fitted (A, tau) for unbinned ML, binned chi2, and binned Poisson ML."""
    n, _ = np.histogram(t, bins=edges)
    # 1. unbinned ML: tau = mean; A = number of events
    unbinned = (t.size, t.mean())
    # 2. binned chi2 with sqrt(n) errors, empty bins dropped
    keep = n > 0
    p_chi2, _ = curve_fit(lambda x, A, tau: mu_bins(A, tau)[keep], centers[keep], n[keep],
                          p0=[n.sum(), 2.0], sigma=np.sqrt(n[keep]), absolute_sigma=True)
    # 3. binned Poisson likelihood
    nll_pois = lambda p: np.sum(mu_bins(*p) - n * np.log(mu_bins(*p)))
    r = minimize(nll_pois, x0=[n.sum(), 2.0], method="Nelder-Mead", bounds=[(1, None), (0.1, 20)])
    return np.array([unbinned, p_chi2, r.x])

t100 = rng.exponential(TAU_MU, 100)
n100, _ = np.histogram(t100, bins=edges)
print("bin counts:", n100)
print("empty bins:", np.sum(n100 == 0))
results = fit_three_ways(t100)
for name, (A, tau) in zip(["unbinned ML", "binned chi2", "binned Poisson ML"], results):
    print(f"{name:>18}:  A = {A:6.1f},  tau = {tau:.3f} us")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(centers, n100, yerr=np.sqrt(n100), fmt="ko", ms=3, label="simulated data")
tt = np.linspace(0, 12, 300)
width = edges[1] - edges[0]
for (A, tau), name in zip(results, ["unbinned ML", r"binned $\chi^2$", "binned Poisson ML"]):
    ax.plot(tt, A * width / tau * np.exp(-tt / tau), label=name)
ax.set_xlabel(r"decay time ($\mu$s)")
ax.set_ylabel(r"decays per 0.3 $\mu$s")
ax.legend()
plt.show()

On a single data set, the three answers differ, but that could be luck. To see whether a method is **systematically** off (biased), you must repeat the experiment many times. That is a **toy Monte Carlo study**: simulate many independent "toy" experiments with known true parameters, fit each one, and look at the distribution of the results. 500 toys take a second or two.

In [ ]:
n_toys = 500
toy_results = np.array([fit_three_ways(rng.exponential(TAU_MU, 100)) for _ in range(n_toys)])
# shape: (toys, method, parameter)

expected_A = 100 * (1 - np.exp(-12 / TAU_MU))      # true decays inside 0-12 us
print(f"true: A = {expected_A:.1f} (inside the histogram range), tau = {TAU_MU}")
for k, name in enumerate(["unbinned ML", "binned chi2", "binned Poisson ML"]):
    A_k, tau_k = toy_results[:, k, 0], toy_results[:, k, 1]
    print(f"{name:>18}: mean A = {A_k.mean():6.1f},  mean tau = {tau_k.mean():.3f} us,  spread of tau = {tau_k.std():.3f} us")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bins_tau = np.linspace(1, 4.5, 50)
for k, name in enumerate(["unbinned ML", r"binned $\chi^2$", "binned Poisson ML"]):
    ax.hist(toy_results[:, k, 1], bins=bins_tau, histtype="step", lw=2, label=name)
ax.axvline(TAU_MU, color="k", ls="--", label="true value")
ax.set_xlabel(r"fitted $\tau$ ($\mu$s)")
ax.set_ylabel("toy experiments")
ax.legend()
plt.show()

(The unbinned "A" is the total number of events, 100, including the few beyond 12 $\mu$s, so compare its $\tau$ only.)

What the toys show:

- **Unbinned ML** and **binned Poisson ML** are centered on the true lifetime and have the smallest spread. Binning into narrow bins loses almost nothing.
- **Binned $\chi^2$** is biased: the normalization comes out too low, the lifetime is pulled away from the truth, and the spread is larger. The reasons are the ones from last week: bins that fluctuate low get small error bars and too much weight, and empty bins are ignored.

The lesson: **with low counts per bin, use a Poisson likelihood (or an unbinned fit), not $\chi^2$.** With many counts per bin (say more than 10–20 everywhere), all methods agree.

## 6. Signal plus background: an unbinned fit with two parameters

Real muon-lifetime experiments also record **background**: random coincidences (for example, two unrelated cosmic rays) that are equally likely at any time in the measurement window $0 \le t \le T$. The PDF of a recorded time is then a mixture:

$$p(t\,|\,\tau, f) = f\,\frac{e^{-t/\tau}}{\tau\,(1 - e^{-T/\tau})} + (1 - f)\,\frac{1}{T},$$

where $f$ is the fraction of events that are real decays. The factor $1 - e^{-T/\tau}$ renormalizes the exponential, because decays after $T$ are not recorded. Each term integrates to 1 over $[0, T]$, so $p$ does too.

Simulate 1000 events with $T = 20\ \mu$s and $f = 0.8$. The signal times are drawn from the **truncated** exponential by inverse transform: $t = -\tau\ln[1 - u(1 - e^{-T/\tau})]$.

In [ ]:
T_win = 20.0              # measurement window, us
f_true = 0.8
N_ev = 1000

def simulate_with_background(n_events, tau, f, rng):
    """Truncated-exponential signal plus flat background in [0, T_win]."""
    n_sig = rng.binomial(n_events, f)                 # number of real decays
    u = rng.random(n_sig)
    sig = -tau * np.log(1 - u * (1 - np.exp(-T_win / tau)))
    bkg = rng.uniform(0, T_win, n_events - n_sig)
    return np.concatenate([sig, bkg])

t_bg = simulate_with_background(N_ev, TAU_MU, f_true, rng)

def pdf_sb(t, tau, f):
    sig = np.exp(-t / tau) / (tau * (1 - np.exp(-T_win / tau)))
    return f * sig + (1 - f) / T_win

def nll_sb(params, t):
    tau, f = params
    return -np.sum(np.log(pdf_sb(t, tau, f)))

In [ ]:
res_sb = minimize(nll_sb, x0=[1.5, 0.5], args=(t_bg,), method="Nelder-Mead",
                  bounds=[(0.1, 20), (0.0, 1.0)])
H_sb = hessian(lambda p: nll_sb(p, t_bg), res_sb.x, h=[1e-3, 1e-3])
cov_sb = np.linalg.inv(H_sb)
err_sb = np.sqrt(np.diag(cov_sb))

print(res_sb.message)
print(f"tau = {res_sb.x[0]:.3f} +/- {err_sb[0]:.3f} us   (true {TAU_MU})")
print(f"f   = {res_sb.x[1]:.3f} +/- {err_sb[1]:.3f}      (true {f_true})")
print(f"correlation(tau, f) = {cov_sb[0, 1] / (err_sb[0] * err_sb[1]):.3f}")

`args=(t_bg,)` passes extra arguments to the function being minimized: `minimize` calls `nll_sb(params, t_bg)`. (The trailing comma makes it a one-element tuple.)

The fit used no histogram at all. To *show* it, though, we draw a histogram and overlay the fitted PDF scaled to expected counts per bin: $N \times p(t) \times \text{bin width}$.

In [ ]:
tau_fit, f_fit = res_sb.x
edges_sb = np.linspace(0, T_win, 51)
w_sb = edges_sb[1] - edges_sb[0]
tt = np.linspace(0, T_win, 400)

fig, ax = plt.subplots(figsize=(6, 4))
n_sb, _, _ = ax.hist(t_bg, bins=edges_sb, histtype="step", color="k", label="simulated data")
ax.plot(tt, N_ev * w_sb * pdf_sb(tt, tau_fit, f_fit), "C1-", lw=2, label="fit: total")
ax.plot(tt, N_ev * w_sb * pdf_sb(tt, tau_fit, f_fit) - N_ev * w_sb * (1 - f_fit) / T_win, "C0--", label="fit: decays")
ax.axhline(N_ev * w_sb * (1 - f_fit) / T_win, color="C2", ls=":", label="fit: background")
ax.set_yscale("log")
ax.set_xlabel(r"time ($\mu$s)")
ax.set_ylabel(r"events per 0.4 $\mu$s")
ax.legend()
plt.show()

On a log scale the exponential is a straight line that flattens out into the background floor, just like the real decay data you fitted last week.

## 7. Toy Monte Carlo: are the uncertainties right?

A quoted uncertainty is a *claim*: "if you repeated this experiment, the result would scatter by about this much, and the interval $\hat\tau \pm \sigma$ would contain the true value about 68% of the time." Toys let you check that claim.

For each toy experiment compute the **pull**

$$g = \frac{\hat\tau - \tau_\text{true}}{\sigma_{\hat\tau}} .$$

If the estimate is unbiased and the uncertainty is right, pulls follow a **standard normal distribution**: a Gaussian with mean 0 and standard deviation 1. Then:

- pull mean $\ne 0$ → bias;
- pull standard deviation $> 1$ → uncertainties underestimated; $< 1$ → overestimated.

**Coverage** is the fraction of toys whose interval contains the truth, i.e. $|g| < 1$. It should be about 68.3%.

Use the simple exponential fit with $N = 50$, where $\hat\tau = \bar t$ and $\sigma = \hat\tau/\sqrt N$ make each toy instant.

In [ ]:
n_toys = 5000
N_toy = 50
toys = rng.exponential(TAU_MU, size=(n_toys, N_toy))   # each row is one experiment
tau_hats = toys.mean(axis=1)
sigmas = tau_hats / np.sqrt(N_toy)
pulls = (tau_hats - TAU_MU) / sigmas

print(f"mean of tau_hat = {tau_hats.mean():.4f} us (true {TAU_MU})")
print(f"spread of tau_hat = {tau_hats.std():.4f} us;  typical quoted sigma = {sigmas.mean():.4f} us")
print(f"pull mean = {pulls.mean():.3f},  pull std = {pulls.std():.3f}")
print(f"coverage (|pull| < 1) = {np.mean(np.abs(pulls) < 1):.3f}   (ideal 0.683)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(tau_hats, bins=60, density=True, alpha=0.6)
axes[0].axvline(TAU_MU, color="k", ls="--", label="true")
axes[0].set_xlabel(r"fitted $\hat\tau$ ($\mu$s)")
axes[0].set_ylabel("probability density")
axes[0].legend()

axes[1].hist(pulls, bins=60, range=(-4, 4), density=True, alpha=0.6, label="toys")
g = np.linspace(-4, 4, 200)
axes[1].plot(g, norm.pdf(g), "k-", label="standard normal")
axes[1].set_xlabel("pull")
axes[1].legend()
plt.tight_layout()
plt.show()

`scipy.stats.norm.pdf(x)` is the standard normal density $e^{-x^2/2}/\sqrt{2\pi}$.

With 50 events the pulls are close to a unit Gaussian, and coverage is close to 68%. Look carefully, though: the $\hat\tau$ distribution has a longer tail to the high side, and the pull distribution is slightly shifted. Both effects grow as $N$ shrinks, which you will explore next. This is exactly how experiments validate their fitting procedures before trusting them on real data.

### Your turn 3

1. Repeat the toy study for $N = 5$, $N = 10$ and $N = 200$ events per experiment. For each, print the pull mean, pull standard deviation, and coverage.
2. For $N = 10$, instead of $\hat\tau \pm \sigma$, use the asymmetric interval from the $\Delta(-2\ln L) = 1$ scan (with `brentq`) for each of 500 toys, and compute its coverage. Which interval has better coverage?

In [ ]:
# Your code here

## 8. The bootstrap

Toys need a model to simulate from. What if you only have your one data set and no reliable model of how it was produced? The **bootstrap** (as in "pulling yourself up by your bootstraps") treats the data set itself as the best available description of the underlying distribution:

1. From your $N$ data points, draw $N$ points **with replacement** (some points get picked twice, some not at all). This is a "bootstrap sample."
2. Compute your estimate on that sample.
3. Repeat many times. The standard deviation of the estimates is the uncertainty.

`rng.choice(data, size=N, replace=True)` draws such a sample. Try it on the 50 decay times, where we know the answer should be close to $\hat\tau/\sqrt N$.

In [ ]:
n_boot = 2000
boot_means = np.array([rng.choice(t_data, size=t_data.size, replace=True).mean() for _ in range(n_boot)])
print(f"bootstrap uncertainty on tau_hat: {boot_means.std():.4f} us")
print(f"likelihood (tau_hat/sqrt(N)):      {sigma_tau:.4f} us")

They agree well. The real power of the bootstrap is for estimates whose uncertainty has no simple formula. For example, the **median** decay time estimates the half-life $t_{1/2} = \tau\ln 2$ directly, without assuming an exponential shape.

In [ ]:
boot_medians = np.array([np.median(rng.choice(t_data, size=t_data.size, replace=True)) for _ in range(n_boot)])
print(f"median of data = {np.median(t_data):.3f} +/- {boot_medians.std():.3f} us")
print(f"tau_hat * ln 2 = {tau_hat * np.log(2):.3f} +/- {sigma_tau * np.log(2):.3f} us  (uses the exponential model)")
print(f"true half-life = {TAU_MU * np.log(2):.3f} us")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(boot_medians, bins=40)
ax.set_xlabel(r"bootstrap median decay time ($\mu$s)")
ax.set_ylabel("bootstrap samples")
plt.show()

The median is a less precise estimator than the model-based $\hat\tau\ln 2$: that is the price of not using the knowledge that the distribution is exponential. (The bootstrap distribution of a median looks spiky because medians can only take values from the data set.)

### Your turn 4

Use the bootstrap to estimate the uncertainty on the background-fraction $f$ from the signal-plus-background fit in Section 6. Use 200 bootstrap samples (each needs a call to `minimize`, so this takes several seconds). Compare with the Hessian uncertainty `err_sb[1]`.

In [ ]:
# Your code here

## 9. Goodness of fit and model comparison

### Goodness of fit

With $\chi^2$, the minimum value told you whether the fit was good. The value of $-2\ln L$ at the minimum of an **unbinned** fit does *not* do this by itself: its size depends on the data in a way that does not test the model shape. The standard approach is to bin the data and compute the **Poisson deviance** (also called the likelihood-ratio $\chi^2$):

$$\chi^2_\lambda = 2\sum_i \left[\mu_i - n_i + n_i\ln\frac{n_i}{\mu_i}\right],$$

with the convention that the logarithm term is 0 when $n_i = 0$. It compares the fitted model with a "perfect" model that predicts every bin exactly, and for a correct model it approximately follows a $\chi^2$ distribution with $\nu = (\text{bins} - \text{parameters})$ degrees of freedom. It behaves well even with low counts, unlike Pearson's $\chi^2$.

Apply it to the signal-plus-background fit, using the 50-bin histogram `n_sb` from the plot above.

In [ ]:
def poisson_deviance(n, mu):
    """2 * sum(mu - n + n ln(n/mu)), with n ln(n/mu) = 0 for empty bins."""
    term = np.zeros_like(mu)
    nz = n > 0
    term[nz] = n[nz] * np.log(n[nz] / mu[nz])
    return 2 * np.sum(mu - n + term)

# expected counts per bin: integrate the fitted PDF over each bin (simple midpoint rule is fine here)
mid = 0.5 * (edges_sb[:-1] + edges_sb[1:])
mu_sb = N_ev * w_sb * pdf_sb(mid, tau_fit, f_fit)
dev = poisson_deviance(n_sb, mu_sb)
ndf = len(n_sb) - 2
print(f"Poisson deviance = {dev:.1f} for {ndf} degrees of freedom;  p-value = {chi2.sf(dev, ndf):.3f}")

(Strictly, the expected count is the integral of the PDF over the bin; the midpoint value times the bin width is accurate here because the bins are narrow compared to $\tau$. Your fits for Project 3 should integrate properly.)

### Comparing two models with $\Delta \ln L$

Is the background really needed? Fit the data again with a **simpler model** that has no background ($f = 1$ fixed) and compare the minimum values of $-2\ln L$. The more flexible model always fits at least as well, so the question is whether the improvement is bigger than chance would produce.

For **nested** models (the simple one is a special case of the complex one), **Wilks' theorem** says: if the simple model is true, the improvement

$$\Delta = (-2\ln L)_\text{simple} - (-2\ln L)_\text{complex}$$

approximately follows a $\chi^2$ distribution with degrees of freedom equal to the number of extra parameters (here 1). A large $\Delta$ with a tiny $p$-value means the simple model is rejected. (A technical footnote: because $f$ cannot exceed 1, the true $p$-value is half of what `chi2.sf` gives; it does not change any conclusion below.)

In [ ]:
def compare_models(t):
    """Return Delta(-2 ln L) between exponential-only and exponential+background fits."""
    r_simple = minimize(lambda p: nll_sb([p[0], 1.0], t), x0=[2.0], method="Nelder-Mead", bounds=[(0.1, 20)])
    r_full = minimize(nll_sb, x0=[2.0, 0.7], args=(t,), method="Nelder-Mead", bounds=[(0.1, 20), (0.0, 1.0)])
    delta = max(2 * (r_simple.fun - r_full.fun), 0.0)   # cannot be negative (guards against round-off)
    return delta, r_simple.x[0], r_full.x

delta, tau_simple, _ = compare_models(t_bg)
print(f"data WITH background:    Delta(-2lnL) = {delta:8.1f},  p = {chi2.sf(delta, 1):.2g},  "
      f"(no-background fit gives tau = {tau_simple:.2f} us)")

t_clean = simulate_with_background(N_ev, TAU_MU, 1.0, rng)     # f = 1: no background at all
delta0, _, full0 = compare_models(t_clean)
print(f"data WITHOUT background: Delta(-2lnL) = {delta0:8.1f},  p = {chi2.sf(delta0, 1):.2g},  "
      f"(full fit gives f = {full0[1]:.3f})")

With background in the data, the simple model is overwhelmingly rejected, and forcing it gives a badly wrong lifetime. Without background, the extra parameter buys almost nothing, and you should prefer the simpler model.

For models that are *not* nested, or to penalize extra parameters more generally, people use criteria such as the Akaike information criterion, AIC $= -2\ln L + 2k$ for $k$ parameters (smaller is better).

### Your turn 5

How small a background can you detect? Simulate data sets of 1000 events with $f = 0.99$ (1% background). For 20 different simulated data sets, compute $\Delta(-2\ln L)$ with `compare_models` and count how many give $p < 0.05$. Repeat for $f = 0.97$. What does this tell you about planning an experiment?

In [ ]:
# Your code here

## Summary

- **Likelihood:** $L(\theta) = \prod_i p(x_i\,|\,\theta)$, the probability of the observed data as a function of the parameters. Maximize it, in practice by minimizing $-\ln L = -\sum_i \ln p(x_i\,|\,\theta)$.
- For Gaussian errors, $-2\ln L = \chi^2 + $ const: least squares is a special case.
- **Exponential lifetime:** $\hat\tau = \bar t$, $\sigma_{\hat\tau} = \hat\tau/\sqrt N$.
- **Uncertainties:** scan $-2\ln L$ and take $\Delta(-2\ln L) = 1$ (can be asymmetric), or invert the **Hessian** of $-\ln L$: $C = H^{-1}$ (`np.linalg.inv`).
- `scipy.optimize.minimize(f, x0, args=(...), method="Nelder-Mead", bounds=[...])`; check `res.success`, use `res.x`, `res.fun`.
- **Binned Poisson likelihood:** $-\ln L = \sum_i (\mu_i - n_i\ln\mu_i)$; handles empty bins. **Binned $\chi^2$ with $\sqrt n$ errors is biased at low counts.** Unbinned fits use all the information.
- **Mixture models** for signal + background: $p = f\,p_\text{sig} + (1-f)\,p_\text{bkg}$, each normalized on the measurement window.
- **Toy Monte Carlo:** simulate many experiments, fit each; check bias (mean), pulls (should be unit Gaussian), and coverage (68.3% within $\pm1\sigma$).
- **Bootstrap:** resample the data with replacement (`rng.choice(data, size=N, replace=True)`); the spread of the estimates is the uncertainty.
- **Goodness of fit:** Poisson deviance on a histogram ~ $\chi^2(\nu)$. **Model comparison:** $\Delta(-2\ln L)$ between nested models ~ $\chi^2$(extra parameters).

## Further reading

- G. Cowan, *Statistical Data Analysis* (Oxford, 1998), chapters 6–7, and his free lecture notes at https://www.pp.rhul.ac.uk/~cowan/stat_course.html
- Particle Data Group, *Review of Particle Physics*, "Statistics" chapter (free at https://pdg.lbl.gov)
- SciPy `minimize` documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
- M. Newman, *Computational Physics*, Chapter 6 (minimization) and Chapter 10 (random processes)